# Lab 7.1 &mdash; The Automation Pipeline

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Walk the six pipeline stages in order, from trigger to act
- See where the Day-3 loop lives (inside reason/draft)
- Mark the approval checkpoint that guards the irreversible act

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; The task-automation pipeline](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-01")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Every task-automation agent, however complex, follows the **same pipeline** (deck slide 5):
**trigger** &rarr; **gather** &rarr; **draft** &rarr; **validate** &rarr; **approve** &rarr; **act**.
The Day-3 ReAct loop lives inside *reason/draft*; the outer stages &mdash; gather, validate,
approve &mdash; are what make it **reliable** and **safe**. The **approve** stage is a human
checkpoint that guards the one irreversible step: **act**. (This lab is pure Python &mdash; it's the
scaffold the real Groq-driven steps in later labs slot into.)

In [ ]:
PIPELINE = ["trigger", "gather", "draft", "validate", "approve", "act"]
print("the shape of every automation:")
print(" -> ".join(PIPELINE))

## Build it
Implement `next_stage` (what follows each stage) and `is_checkpoint` (the human gate before the
irreversible act); `run_pipeline` then walks the whole thing.

In [ ]:
PIPELINE = ["trigger", "gather", "draft", "validate", "approve", "act"]

def next_stage(current):
    i = PIPELINE.index(current)
    return ___   # TODO: "done" if this is the last stage, else the next stage

def is_checkpoint(stage):
    # the human approval gate: a person must approve before the irreversible act
    return ___   # TODO: True only for the "approve" stage

def run_pipeline():
    order, stage = [], "trigger"
    while stage != "done":
        order.append(stage)
        stage = next_stage(stage)
    return order

try:
    print("after trigger ->", next_stage("trigger"))
    print("after act     ->", next_stage("act"))
    print("full run:", run_pipeline())
    print("checkpoint at approve?", is_checkpoint("approve"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `run_pipeline()` walks all six stages in order &mdash; that ordering is the contract every later lab honours.
- `is_checkpoint("approve")` marks the one **human** gate; everything before it is autonomous, everything after it is irreversible.
- The `draft` stage is where the real model runs (Lab 6); `gather` is where real tools run (Lab 2).

## Your turn (open task &mdash; no grader)
Add a seventh stage &mdash; e.g. `"log"` after `act` &mdash; and re-run `run_pipeline()`. **What good looks
like:** the walk includes your new stage in the right place and `next_stage` still terminates at `"done"`.
Then ask yourself: which of your stages are reversible, and which one truly needs the human gate?

---
### Key takeaway
Trigger -> gather -> draft -> validate -> approve -> act. The outer stages are what turn a demo agent into an automation. Next: the gather stage -- grounding the task in real data with real tools.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>